# Groq Prompt Filtering

In [7]:
import os
import sys
sys.path.insert(0, ".")

from dotenv import load_dotenv
from groq import Groq
from src.common.data import create_fixed_splits, load_bnsentmix
from src.common.io_utils import batch_sentences, append_jsonl
from src.filtering.prompts import decisions_path
from src.filtering.prompts import build_batch_prompt, parse_batch_response, prompt_version_label
PROMPT_VERSION = "v3"

load_dotenv(override=True)
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL_NAME = "qwen/qwen3.8-27b"

In [8]:
def classify_batch_with_groq(sentences, max_completion_tokens=512):
    response = groq_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": build_batch_prompt(sentences, PROMPT_VERSION)}],
        temperature=0,
        reasoning_effort="none",
        reasoning_format="hidden",
        max_completion_tokens=max_completion_tokens,
    )

    return parse_batch_response(
        response.choices[0].message.content,
        expected_count=len(sentences),
    )

# Test One Shared Batch

In [9]:
splits = create_fixed_splits(load_bnsentmix())
batch = next(batch_sentences(splits["train"], batch_size=5))
response = groq_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": build_batch_prompt(batch, PROMPT_VERSION)}],
    temperature=0,
    reasoning_effort="none",
    reasoning_format="hidden",
    max_completion_tokens=512,
)
decisions = parse_batch_response(response.choices[0].message.content, len(batch))
print(decisions)

[True, True, True, True, True]


# Run a Controlled 50-Sentence Batch

In [10]:
pilot_batch = next(batch_sentences(splits["train"], batch_size=50))
pilot_decisions = classify_batch_with_groq(pilot_batch)

print("Processed:", len(pilot_decisions))
print("Kept:", sum(pilot_decisions))
print("Removed:", len(pilot_decisions) - sum(pilot_decisions))

Processed: 50
Kept: 50
Removed: 0


# Run the Full Groq Training Filter

This processes only the 14,010 training rows. The validation and test sets are never sent to the filter. Results are checkpointed to JSONL so an interrupted run can resume.

In [11]:
import json
import time
from groq import RateLimitError, APIError

BATCH_SIZE = 50
DECISIONS_PATH = decisions_path("groq", MODEL_NAME, PROMPT_VERSION)
DECISIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

completed_indices = set()
if DECISIONS_PATH.exists():
    with DECISIONS_PATH.open(encoding="utf-8") as file:
        for line in file:
            if line.strip():
                completed_indices.add(json.loads(line)["sentence_index"])

print("Result file:", DECISIONS_PATH)
print("Already completed:", len(completed_indices))

Result file: D:\Research\Sentiment-Analysis\outputs\groq_qwen_qwen3.8-27b_v3.jsonl
Already completed: 11050


In [12]:
for start in range(0, len(splits["train"]), BATCH_SIZE):
    stop = min(start + BATCH_SIZE, len(splits["train"]))
    indices = list(range(start, stop))
    pending_indices = [index for index in indices if index not in completed_indices]

    if not pending_indices:
        continue

    sentences = [splits["train"][index]["Sentence"] for index in pending_indices]
    decisions = None

    for attempt in range(5):
        try:
            decisions = classify_batch_with_groq(sentences)
            break
        except RateLimitError:
            if attempt == 4:
                raise
            wait_seconds = 30 * (attempt + 1)
            print(f"Rate limit reached; retrying in {wait_seconds} seconds...")
            time.sleep(wait_seconds)
        except APIError:
            if attempt == 4:
                raise
            time.sleep(10 * (attempt + 1))

    records = [
        {
            "sentence_index": index,
            "keep": keep,
            "provider": "groq",
            "model": MODEL_NAME,
            "prompt_version": prompt_version_label(PROMPT_VERSION),
        }
        for index, keep in zip(pending_indices, decisions)
    ]
    append_jsonl(records, DECISIONS_PATH)
    completed_indices.update(pending_indices)
    print(f"Completed {len(completed_indices)}/{len(splits['train'])}")

Completed 11100/14010
Completed 11150/14010
Completed 11200/14010
Completed 11250/14010
Completed 11300/14010
Completed 11350/14010
Completed 11400/14010
Completed 11450/14010
Completed 11500/14010
Completed 11550/14010
Completed 11600/14010
Completed 11650/14010
Completed 11700/14010
Completed 11750/14010
Completed 11800/14010
Completed 11850/14010
Completed 11900/14010
Completed 11950/14010
Completed 12000/14010
Completed 12050/14010
Completed 12100/14010
Completed 12150/14010
Completed 12200/14010
Completed 12250/14010
Completed 12300/14010
Completed 12350/14010
Completed 12400/14010
Completed 12450/14010
Completed 12500/14010
Completed 12550/14010
Completed 12600/14010
Completed 12650/14010
Completed 12700/14010
Completed 12750/14010
Completed 12800/14010
Completed 12850/14010
Completed 12900/14010
Completed 12950/14010
Completed 13000/14010
Completed 13050/14010
Completed 13100/14010
Completed 13150/14010
Completed 13200/14010
Completed 13250/14010
Completed 13300/14010
Completed 